In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

print("torch:", torch.__version__)
print("cuda :", torch.cuda.is_available())

torch: 2.13.0+cu130
cuda : True


In [2]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (..., seq_len_q, d_k)
    K: (..., seq_len_k, d_k)
    V: (..., seq_len_k, d_v)
    mask: (..., seq_len_q, seq_len_k), 1 表示可见, 0 表示屏蔽
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = attn_weights @ V
    return output, attn_weights

In [3]:
batch, seq_len, d_model, num_heads = 2, 5, 512, 8
d_k = d_model // num_heads
print("每个头的维度 d_k =", d_k)

x = torch.randn(batch, seq_len, d_model)
print("原始              :", x.shape)

# 第一步: 拆分 d_model → (num_heads, d_k)
x1 = x.view(batch, seq_len, num_heads, d_k)
print("拆分后            :", x1.shape)

# 第二步: 把 num_heads 挪到第 1 维
x2 = x1.transpose(1, 2)
print("转置后(送进注意力) :", x2.shape)

# ---- 假装这里算完了注意力 ----

# 第三步: 转置回来
x3 = x2.transpose(1, 2)
print("转置回来          :", x3.shape)

# 第四步: 合并 (num_heads, d_k) → d_model
x4 = x3.contiguous().view(batch, seq_len, d_model)
print("合并后            :", x4.shape)

print("和原始一致吗      :", torch.allclose(x, x4))

每个头的维度 d_k = 64
原始              : torch.Size([2, 5, 512])
拆分后            : torch.Size([2, 5, 8, 64])
转置后(送进注意力) : torch.Size([2, 8, 5, 64])
转置回来          : torch.Size([2, 5, 8, 64])
合并后            : torch.Size([2, 5, 512])
和原始一致吗      : True


In [5]:
try:
    bad = x3.view(batch, seq_len, d_model)      # 少了 .contiguous()
except RuntimeError as e:
    print("报错了，内容是:")
    print(e)

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        """(batch, seq_len, d_model) -> (batch, num_heads, seq_len, d_k)"""
        batch, seq_len, _ = x.size()
        x = x.view(batch, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        """(batch, num_heads, seq_len, d_k) -> (batch, seq_len, d_model)"""
        batch, _, seq_len, _ = x.size()
        x = x.transpose(1, 2).contiguous()
        return x.view(batch, seq_len, self.d_model)

    def forward(self, query, key, value, mask=None):
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))

        if mask is not None:
            mask = mask.unsqueeze(1)      # 给 num_heads 维留位置

        out, attn = scaled_dot_product_attention(Q, K, V, mask)
        out = self.combine_heads(out)
        return self.W_o(out), attn

In [7]:
mha = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(2, 5, 512)

out, attn = mha(x, x, x)
print("输出形状:", out.shape)          # (2, 5, 512) —— 和输入一样
print("权重形状:", attn.shape)         # (2, 8, 5, 5) —— 多出来的 8 就是头

print("\n每个头每行之和:")
print(attn.sum(dim=-1))                # 8×5 = 40 行，全应为 1

total = sum(p.numel() for p in mha.parameters())
print("\n总参数量:", total)             # 1,050,624

# 拆开验证参数量的来源
print("W_q.weight:", mha.W_q.weight.shape, "=", mha.W_q.weight.numel())
print("W_q.bias  :", mha.W_q.bias.shape,   "=", mha.W_q.bias.numel())
print("单层合计  :", mha.W_q.weight.numel() + mha.W_q.bias.numel())
print("四层合计  :", (mha.W_q.weight.numel() + mha.W_q.bias.numel()) * 4)

输出形状: torch.Size([2, 5, 512])
权重形状: torch.Size([2, 8, 5, 5])

每个头每行之和:
tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)

总参数量: 1050624
W_q.weight: torch.Size([512, 512]) = 262144
W_q.bias  : torch.Size([51

In [8]:
seq_len = 5
causal = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)

out_m, attn_m = mha(x, x, x, mask=causal)
print("加 mask 后每行仍为 1:", attn_m.sum(dim=-1)[0, 0])

for h in range(3):
    print(f"\n--- head {h} ---")
    print(attn_m[0, h])

加 mask 后每行仍为 1: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SelectBackward0>)

--- head 0 ---
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6183, 0.3817, 0.0000, 0.0000, 0.0000],
        [0.3142, 0.4710, 0.2148, 0.0000, 0.0000],
        [0.1812, 0.2707, 0.3543, 0.1938, 0.0000],
        [0.1097, 0.3349, 0.0939, 0.1968, 0.2646]], grad_fn=<SelectBackward0>)

--- head 1 ---
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4369, 0.5631, 0.0000, 0.0000, 0.0000],
        [0.3386, 0.4209, 0.2405, 0.0000, 0.0000],
        [0.1757, 0.3182, 0.2502, 0.2558, 0.0000],
        [0.2799, 0.1193, 0.0980, 0.2617, 0.2410]], grad_fn=<SelectBackward0>)

--- head 2 ---
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4969, 0.5031, 0.0000, 0.0000, 0.0000],
        [0.3055, 0.3127, 0.3818, 0.0000, 0.0000],
        [0.2383, 0.4026, 0.1844, 0.1747, 0.0000],
        [0.2020, 0.2482, 0.1839, 0.2297, 0.1362]], grad_fn=<SelectBackward0>)


In [9]:
lang = torch.randn(2, 5, 512)       # 5 个语言 token
img  = torch.randn(2, 196, 512)     # 196 个图像 patch

out_c, attn_c = mha(lang, img, img)
#                    ↑Q    ↑K   ↑V

print("输出形状:", out_c.shape)      # (2, 5, 512) —— 长度跟随 Q
print("权重形状:", attn_c.shape)     # (2, 8, 5, 196) —— 不是方阵!
print("每行之和:", attn_c.sum(dim=-1)[0, 0])

输出形状: torch.Size([2, 5, 512])
权重形状: torch.Size([2, 8, 5, 196])
每行之和: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SelectBackward0>)


In [10]:
mha1 = MultiHeadAttention(d_model=512, num_heads=1)
x_small = torch.randn(1, 4, 512)

out_mha, attn_mha = mha1(x_small, x_small, x_small)

# 手动走一遍单头流程
Q = mha1.W_q(x_small)
K = mha1.W_k(x_small)
V = mha1.W_v(x_small)
out_manual, attn_manual = scaled_dot_product_attention(Q, K, V)
out_manual = mha1.W_o(out_manual)

print("输出一致:", torch.allclose(out_mha, out_manual, atol=1e-6))
print("权重一致:", torch.allclose(attn_mha.squeeze(1), attn_manual, atol=1e-6))

输出一致: True
权重一致: True
